In [39]:
from datetime import datetime, time
from Database.TPData import TPData, TPDataDa
from OrderBook.OrderBook import OrderBookSnaps
from SynthSpread.spreadviewer_class import SpreadSingle
from Strategies.Sparse_momentum.ob_attributes import OB_attributes, TR_attributes
import pandas as pd
import numpy as np
from Utilities.excel_loaders import conn_out_xload
from Utilities.dfutils import dict_iloc
from Utilities.Storage import get_curr_storage_path
from Utilities.func_utils import load_arguments
import pickle
import mplfinance as mpf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import datetime as dt

In [40]:
STORAGE_ = get_curr_storage_path()
l_path = '//192.168.10.91/data/Data/orderbooks/base/'

def load_ob(m, t, dt, p_d, bT, eT):
    ob_class = OrderBookSnaps(verbose=True)
    file_path = l_path + t.split('_')[0] + '/'
    file_name = m + '_' + t.split('_')[0] + '_' + p_d.strftime('%y%m%d') + '_' + dt.strftime('%y%m%d') + '.p'
    print('%s Loading OrderBook %d...' % (dt.strftime('%y-%m-%d'), 0))
    print(file_path + file_name)
    time_load = ob_class.import_data(file_path + file_name)
    print('OrderBook %d created in %d sec' % (0, time_load))
    # ob_class.LoB_truncate(thres_vol=1)
    return ob_class.LoB_select(bT, eT, freq=None)

def variables_from_instrument(instrument: str):
    result = {
        'mkt': None,
        'tenor': None,
        'tn': None
    }
    for x in ['de', 'fr', 'ttf']:
        if x in instrument:
            result['mkt'] = x
    result['tenor'] = instrument[-2]
    result['tn'] = int(instrument[-1])
    return result

In [41]:
dates_out = conn_out_xload()
allwd_broker_ids = [1441]

# ------------------ dataset prep ---------------------------------

# Load arguments
_INSTRUMENTS = ['dey1']
_START_DATE, _END_DATE = '2025-03-01', '2025-04-15'
ins_dicts = [variables_from_instrument(x) for x in _INSTRUMENTS]
n_s = 2
mkt_list = [ins_dict['mkt'] for ins_dict in ins_dicts]
tenor_list = [ins_dict['tenor'] for ins_dict in ins_dicts]
tn1_list = [ins_dict['tn'] for ins_dict in ins_dicts]
ts_lag = (lambda i: mkt_list[i] + tenor_list[i] + str(tn1_list[i]))(0)

tn2_list = []
prod = 'base'
venue_list = ['eex']
start_date = datetime.strptime(_START_DATE, '%Y-%m-%d').date()
end_date = datetime.strptime(_END_DATE, '%Y-%m-%d').date()

if not tn2_list:
    tn_list = [str(t1) for t1 in tn1_list]
else:
    tn_list = [str(t1) + '_' + str(t2) for (t1, t2) in zip(tn1_list, tn2_list)]

dates = pd.date_range(start_date, end_date, freq='B')

spread_class = SpreadSingle(mkt_list, tenor_list, tn1_list, tn2_list, venue_list)
product_date1 = spread_class.product_dates(dates, n_s, tn_bool=True)
product_date2 = spread_class.product_dates(dates, n_s, tn_bool=False)

start_time = time(9, 0, 0, 0)
end_time = time(17, 40, 0, 0)

gran = None

is_db = True
is_tr = True

if is_db:
    data_class = TPData() 
else:
    data_class = TPDataDa()



data_class.create_connection('OracleSQL')
inst_trades = data_class.get_trades_inst('de', venue_list, start_date, end_date, prod='base', spread_bool=False)
instrument_ts = inst_trades[inst_trades.eval('broker_id==1441')].index.drop_duplicates()


obAtt = OB_attributes(['b_price', 'a_price'])
df_orders = {}
df_trades = {}

for k, ds in enumerate(dates):
    if ds in dates_out:
        continue
    bT = datetime.combine(ds, start_time)
    eT = datetime.combine(ds, end_time)
    pd1_aux = [None if p is None else p[k] for p in product_date1]
    pd2_aux = [None if p is None else p[k] for p in product_date2]
    for (m, t, n, pd1, pd2) in zip(mkt_list, tenor_list, tn_list,
                                                pd1_aux, pd2_aux):
        i = m + t + str(n)
        # Order Book attributes
        ob_class = OrderBookSnaps(verbose=True)
        LoB, ts = load_ob(m, t, bT, pd1, bT, eT)
        ob_class.update_data(LoB, ts)
        orders = obAtt.prepare_ob_data(LoB, [0], aonn=True)
        # Trades
        data_class.create_connection('OracleSQL')
        trades = data_class.get_trades(m, t, venue_list, pd1, bT, eT,
                                            prod)
        trades = trades[trades['broker_id'].isin(allwd_broker_ids)]
        trades = trades.reset_index(names='datetime').drop_duplicates('datetime', keep='last').set_index('datetime')
        trades = data_class.clean_trades(trades, pd.DataFrame(orders), is_verbose=True)

        if i not in df_trades:
            df_trades[i] = trades
            df_orders[i] = pd.DataFrame(orders).set_index('timestamp')
        else:
            df_trades[i] = pd.concat([df_trades[i], trades])
            df_orders[i] = pd.concat([df_orders[i], pd.DataFrame(orders).set_index('timestamp')])

Connected to the database oracle
Disconnected from the database oracle
25-03-03 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_260101_250303.p
OrderBook 0 created in 24 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [price, volume, action, broker_id, own_trades]
Index: []


25-03-04 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_260101_250304.p
OrderBook 0 created in 27 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [price, volume, action, broker_id, own_trades]
Index: []


25-03-05 Loading OrderBook 0...
//192.168.10.91/data/Data/orderbooks/base/y/de_y_260101_250305.p
OrderBook 0 created in 71 sec
Connected to the database oracle
Disconnected from the database oracle
Deleting following trades 

Empty DataFrame
Columns: [price, volume, action, broker_id, own_trades]
Index: []



In [42]:
df_trades['dey1']

,price,volume,action,broker_id,own_trades
datetime,,,,,
2025-03-03 09:00:57.495030581,87.00,1,1,1441,False
2025-03-03 09:00:57.612058659,87.00,1,1,1441,False
2025-03-03 09:00:58.218863115,87.00,1,1,1441,False
2025-03-03 09:00:58.555367920,87.00,1,1,1441,False
2025-03-03 09:01:08.109247281,87.00,1,1,1441,False
...,...,...,...,...,...
2025-04-15 17:23:51.183619653,82.21,1,-1,1441,False
2025-04-15 17:24:15.253523696,82.30,1,-1,1441,False
2025-04-15 17:25:37.371471619,82.21,2,-1,1441,False


In [ ]:
def create_ohlc_candles(df, trd_col = 'price', time_intervals=['1H', '15T', '5T']):
    """
    Create OHLC candles for specified time intervals from a series of trade data.
 
    Parameters:
    - df (pd.DataFrame): DataFrame with 'timestamp' as the index and 'price' column representing trade prices.
    - time_intervals (list): List of strings representing time intervals, e.g., ['1H', '15T', '5T'].
 
    Returns:
    - dict of pd.DataFrame: Dictionary where each key is a time interval and the value is the OHLC DataFrame for that interval.
    """
    ohlc_data = {}
 
    # Ensure 'timestamp' is the index and in datetime format
    df.index = pd.to_datetime(df.index)
 
    for interval in time_intervals:
        ohlc_df = df[trd_col].resample(interval).ohlc()
        ohlc_data[interval] = ohlc_df
 
    return ohlc_data
 
# Example usage
# Suppose 'df' is your DataFrame with trade data, indexed by 'timestamp' and containing a 'price' column

 


C:\Users\krajcovic\AppData\Local\Temp\ipykernel_33424\453607027.py:18: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [56]:
interval = '2H'

ohlc_data = create_ohlc_candles(df_trades['dey1'], time_intervals=[interval])

df = ohlc_data[interval].copy()

df = df.reset_index().dropna()

df['min'] = np.nan
df['max'] = np.nan

for idx, row in df.iterrows():
    index_placeholder, o,h,l,c, df_min, df_max = row
    if index_placeholder == dt.datetime(2023,2,25):
        print('stop')
    if idx == 0:
        last_min = [idx,l]
        last_min2 = [idx,l]
        last_max = [idx,h]
        last_max2 = [idx,h]
        df.loc[idx, 'min'] = l
        df.loc[idx, 'max'] = h
        continue

    last_max_idx_diff = idx - last_max[0]
    last_max2_idx_diff = idx - last_max2[0]
    last_min_idx_diff = idx - last_min[0]
    last_min2_idx_diff = idx - last_min2[0]
    # If the value is maximum
    if last_max[1] < h:
        # if last_max2[1] < h:
        df.loc[idx, 'max'] = h     
        if last_max_idx_diff > 1:
            # Move maxes
            last_max2 = last_max
            last_max = [idx, h]
            # Find new minimum
            # if last_max2[1] < h:
            new_min_slice = df.loc[last_max2[0]:last_max[0], 'low']
            new_min = new_min_slice.min()
            new_min_idx = new_min_slice.idxmin()
            last_min2 = last_min
            last_min = [new_min_idx, new_min]
            df.loc[idx, 'min'] = new_min
        else:
            last_max = [idx, h]

    # If the value is minimum
    if last_min[1] > l:
        # if last_min2[1] > l:
        df.loc[idx,'min'] = l
        if last_min_idx_diff > 1:
            # Move mins
            last_min2 = last_min
            last_min = [idx, l]
            # Find new minimum
            # if last_min2[1] > l:
            new_max_slice = df.loc[last_min2[0]:last_min[0], 'high']
            new_max = new_max_slice.max()
            new_max_idx = new_max_slice.idxmax()
            last_max2 = last_max
            last_max = [new_max_idx, new_max]
            df.loc[idx, 'max'] = new_max
        else:
            last_min = [idx,l]
    

df = df.set_index('datetime')


C:\Users\krajcovic\AppData\Local\Temp\ipykernel_33424\453607027.py:18: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [98]:
import plotly.graph_objects as go

def plot_ohlc_with_extremes(df, title="OHLC Candlestick Chart with Highs and Lows"):
    """
    Plot an interactive Plotly candlestick chart with overlaid min/max markers.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must be indexed by datetime and have columns:
        ['open', 'high', 'low', 'close', 'min', 'max'].
        'min'/'max' may contain NaNs for non‑extrema.
    title : str, optional
        Chart title (default: "OHLC Candlestick Chart with Highs and Lows").
    """
    # 1) Base candlestick
    fig = go.Figure(data=[go.Candlestick(
        x=df.index,
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='OHLC'
    )])

    # 2) Highs
    highs = df['max'].dropna()
    fig.add_trace(go.Scatter(
        x=highs.index,
        y=highs.values,
        mode='markers',
        marker=dict(symbol='x', color='blue', size=8),
        name='Highs'
    ))

    # 3) Lows
    lows = df['min'].dropna()
    fig.add_trace(go.Scatter(
        x=lows.index,
        y=lows.values,
        mode='markers',
        marker=dict(symbol='x', color='orange', size=8),
        name='Lows'
    ))

    # 4) Layout & interaction
    fig.update_layout(
        title=title,
        xaxis_title="Time",
        yaxis_title="Price",
        dragmode='zoom',                 # box‑zoom both axes by default
        xaxis=dict(rangeslider=dict(visible=False))
    )

    # 5) Show with scroll‑wheel zoom and extra zoom buttons
    fig.show(config={
        'scrollZoom': True,
        'modeBarButtonsToAdd': [
            'zoom2d',   # box zoom
            'zoomX',    # X‑axis only
            'zoomY',    # Y‑axis only
            'resetScale2d'
        ]
    })


In [59]:
import pandas as pd
import numpy as np
from datetime import datetime, time
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
from Database.TPData import TPData, TPDataDa
from Math.accumfeatures import EMA, MA, MSTD

In [103]:
n_s = 3
start_date = datetime(2025, 4, 1)
end_date = datetime(2025, 4, 14)
dates = pd.date_range(start_date, end_date, freq='B')
market = ['de']*2
tenor = ['m', 'm']
tn1_list = [1,2]
tn2_list = []
brk_list = ['eex']*2
mm_bool = [True]*2

tau = 5
margin = .43

start_time = time(9, 0, 0, 0)
end_time = time(17, 25, 0, 0)
gran = None
gran_s = '1h'
coeff_list = norm_coeff([1,-1], market)

eql_p = -6.25
w = 0

add_trades = True
ob_data = False


spread_class = SpreadSingle(market, tenor, tn1_list, tn2_list, brk_list)
data_class = SpreadViewerData()
db_class = TPDataDa()
tenors_list = spread_class.tenors_list
if not ob_data:
    data_class.load_best_order_otc(market, tenors_list,
                                   spread_class.product_dates(dates, n_s),
                                   db_class,
                                   start_time=start_time, end_time=end_time)
else:
    # data_class.load_best_ob(market, tenors_list, dates, spread_class.product_dates(dates, n_s),
    #                         v_thres=5, freq=gran)
    data_class.load_best_ob_tp(market, tenors_list,
                               spread_class.product_dates(dates, n_s),
                               db_class,
                               start_time=start_time, end_time=end_time)
if add_trades:
    data_class_tr = SpreadViewerData()
    data_class_tr.load_trades_otc(market, tenors_list, db_class,
                                  start_time=start_time, end_time=end_time)
    
sm_all = pd.DataFrame([])
tm_all = pd.DataFrame([])
for d in dates:
    d_range = pd.date_range(d, d)
    data_dict = spread_class.aggregate_data(data_class, d_range, n_s, gran=gran,
                                            start_time=start_time, end_time=end_time)
    sm = spread_class.spread_maker(data_dict, coeff_list, trade_type=['cmb', 'cmb']).dropna()
    
    sm_all = pd.concat([sm_all, sm], axis=0)
    
    # em = calc_ema_m(sm, tau, margin, w, eql_p)
    # sm = pd.concat([sm, em], axis=1)
    
    # ax = sm.plot(grid=True, legend=True, figsize=(34, 22), title=d.strftime('%Y/%B/%d-%a'))
    if add_trades:
        col_list=['bid', 'ask', 'volume', 'broker_id']
        trade_dict = spread_class.aggregate_data(data_class_tr, d_range, n_s, gran=gran_s,
                                                 start_time=start_time, end_time=end_time,
                                                 col_list=col_list, data_dict=data_dict)
        tm = spread_class.add_trades(data_dict, trade_dict, coeff_list, mm_bool)
        
        tm_all = pd.concat([tm_all, tm], axis=0)

Loading data...

BO for market & date: de // m_1 & 2025-04-01 09:00:00 // 2025-04-01 17:25:00 

https://referencedata.trayport.com/instruments
Duration: 0.708s
https://analytics.trayport.com/api/orders/book?from=2025-04-01T07%3A00%3A00Z&until=2025-04-01T15%3A25%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.190s
Data load completed in 5 sec.

Loading data...

BO for market & date: de // m_2 & 2025-04-01 09:00:00 // 2025-04-01 17:25:00 

https://referencedata.trayport.com/instruments
Duration: 1.128s
https://analytics.trayport.com/api/orders/book?from=2025-04-01T07%3A00%3A00Z&until=2025-04-01T15%3A25%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.766s
Data load completed in 6 sec.

Loading data...

Trades for market & date: de // m_1 & 2025-04-01 09:00:00 // 2025

In [104]:
def calc_ema_m(df_data, tau, margin, w, eql_p):
    mid_ser = .5 * (df_data.loc[:, 'bid'] + df_data.loc[:, 'ask'])
    mid_list = mid_ser.values
    dif_list = [0.001]
    dif_list.extend([abs(x - xl) for x, xl in zip(mid_list[1:], mid_list[:-1])])
    model = EMA(tau, mid_list[0])
    ema_list = [model.push(x, dx) for x, dx in zip(mid_list, dif_list)]
    ema_list = [w * eql_p + (1 - w) * x for x in ema_list]
    bands = [[x - margin, x, x + margin] for x in ema_list]
    return pd.DataFrame(bands, index=mid_ser.index)

def adjust_trds(df_tr, df_em):
    timestamp = df_tr.index
    ts_new = df_em.index.union(timestamp)
    df_em = df_em.reindex(ts_new).ffill().reindex(timestamp)
    lb = df_em.iloc[:, 0]
    ub = df_em.iloc[:, 2]
    df_tr.loc[df_tr['buy'] > ub, 'buy'] = np.nan
    df_tr.loc[df_tr['sell'] < lb, 'sell'] = np.nan
    return df_tr.dropna(how='all')

In [105]:
em = calc_ema_m(sm_all, tau, margin, w, eql_p)
sm = pd.concat([sm_all, em], axis=1)
sm.columns = ['bid', 'ask', 'bid2', 'mid', 'ask2']
sm.index.name = 'datetime'

tm_ = adjust_trds(tm_all, em)

In [106]:
def compute_ohlc_extremes(trades_series,
                          interval: str,
                          trd_col: str) -> pd.DataFrame:
    """
    Given a time‑indexed trades_series (e.g. df_trades['dey1'])
    and a resample interval (e.g. '2H', '5T', etc.),
    this will:
      1) build OHLC candles for that interval via create_ohlc_candles()
      2) walk through each candle and mark local extrema in 'min'/'max'
      3) return a DataFrame indexed by datetime with columns:
         ['open','high','low','close','min','max']
    """
    # 1) build the raw OHLC
    ohlc_dict = create_ohlc_candles(trades_series, time_intervals=[interval],
                                    trd_col=trd_col)
    df = ohlc_dict[interval].copy()

    # 2) prep
    df = df.reset_index().dropna().copy()
    df['min'] = np.nan
    df['max'] = np.nan

    # 3) iterate & compute swings
    last_min  = last_min2  = [0, 0]
    last_max  = last_max2  = [0, 0]

    for idx, row in df.iterrows():
        # unpack
        _, o, h, l, c, _, _ = row

        if idx == 0:
            # init with first candle
            last_min  = last_min2  = [idx, l]
            last_max  = last_max2  = [idx, h]
            df.loc[idx, 'min']  = l
            df.loc[idx, 'max']  = h
            continue

        # distances since previous extremum
        dm1 = idx - last_min[0]
        dm2 = idx - last_min2[0]
        dM1 = idx - last_max[0]
        dM2 = idx - last_max2[0]

        # --- new maximum? ---
        if h > last_max[1]:
            df.loc[idx, 'max'] = h
            if dM1 > 1:
                last_max2 = last_max
                last_max  = [idx, h]
                # find low between last two maxes
                low_slice    = df.loc[last_max2[0]:last_max[0], 'low']
                new_min_val  = low_slice.min()
                new_min_idx  = low_slice.idxmin()
                last_min2    = last_min
                last_min     = [new_min_idx, new_min_val]
                df.loc[idx, 'min'] = new_min_val
            else:
                last_max = [idx, h]

        # --- new minimum? ---
        if l < last_min[1]:
            df.loc[idx, 'min'] = l
            if dm1 > 1:
                last_min2 = last_min
                last_min  = [idx, l]
                # find high between last two mins
                high_slice   = df.loc[last_min2[0]:last_min[0], 'high']
                new_max_val  = high_slice.max()
                new_max_idx  = high_slice.idxmax()
                last_max2    = last_max
                last_max     = [new_max_idx, new_max_val]
                df.loc[idx, 'max'] = new_max_val
            else:
                last_min = [idx, l]

    # 4) restore datetime index and return
    df = df.set_index('datetime')
    return df

In [107]:
df = compute_ohlc_extremes(sm[['mid']], interval='30T', trd_col='mid')

C:\Users\krajcovic\AppData\Local\Temp\ipykernel_33424\453607027.py:18: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.



In [108]:
# assuming `result` is your df from compute_ohlc_extremes()
plot_ohlc_with_extremes(df, title="2‑Hour OHLC with Swing Extremes")


In [ ]:
def calc_ev(df_trades, trd_col, periods=100):
    bin_edges = np.arange(-3.0, 3.25, 0.25)  # Adding 0.05 to include 1.0 as the
    histograms = []
    volatility_list = []
    for i, row in df_trades.reset_index().iterrows():
        curr_price = row[trd_col]
        try:
            data_hist = df_trades[trd_col].values[i-100:i]
            data = df_trades[trd_col].values[i:i+100]
        except IndexError:
            break
        data = data - curr_price
        volatility = np.std(data_hist)
        data = data/volatility
        counts, _ = np.histogram(data, bins=bin_edges, density=False)
        probabilities = counts / np.sum(counts)
        histograms.append(probabilities)
        volatility_list.append(volatility)
 
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
 
    EV = [np.sum(bin_centers * probabilities) for probabilities in histograms]
 
    return 
 

In [116]:
test = calc_ev(sm, trd_col='mid')

In [119]:
len(test)

123003

In [120]:
len(sm)

123003